# Echo (EchoNet-Dynamic) — ejection fraction regression

**Train-from-scratch, conditional.** Per `docs/04-data-models.md`, no verified
Echo-Vision-FM checkpoint was found on Hugging Face at planning time — **that's
worth re-checking before you run this**, since this space moves fast. If
nothing suitable turns up, this notebook trains a compact video model on
**EchoNet-Dynamic** to predict **ejection fraction (EF)** — the task
`docs/12` calls out for this vertical.

**What this notebook does:**
1. Loads EchoNet-Dynamic's `FileList.csv` (per-video EF labels + the official
   train/val/test split column).
2. Samples a fixed number of frames per echo video (full-video 3D CNN training
   is expensive; frame sampling + temporal pooling is the practical choice for
   a single Kaggle GPU).
3. Trains a CNN encoder (per-frame) + temporal pooling head for **EF
   regression** (mean absolute error).
4. Evaluates with MAE and R² against the official test split.

## Kaggle setup
- **Add data:** search *Add Data* for **"EchoNet"** or **"EchoNet-Dynamic"**.
  You need `FileList.csv` plus the `Videos/` folder (`.avi` files). The
  official source requires a Stanford data-use agreement — see
  [echonet.github.io/dynamic](https://echonet.github.io/dynamic/); several
  Kaggle mirrors also exist (`docs/04` flags these as unofficial re-uploads —
  fine for prototyping, use the official source before anything production-relevant).
- **Accelerator:** GPU T4 x2.
- **Internet:** on (for `pip install opencv-python-headless` if not preinstalled).

In [ ]:
# Cell 1 — install packages (opencv is usually preinstalled on Kaggle; this is
# a no-op if so).
!pip install -q opencv-python-headless

In [ ]:
# Cell 2 — imports and reproducibility.
import glob
import json
import os
import random

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tv_models
from sklearn.metrics import mean_absolute_error, r2_score
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Locate the dataset

Searches `/kaggle/input` for `FileList.csv` so this works across mirrors
without hardcoding a dataset slug.

In [ ]:
# Cell 3 — auto-detect the EchoNet-Dynamic root.
candidates = glob.glob("/kaggle/input/**/FileList.csv", recursive=True)
if not candidates:
    raise FileNotFoundError(
        "Could not find FileList.csv under /kaggle/input. "
        "Add an EchoNet-Dynamic dataset via 'Add Data', or set ECHONET_ROOT manually."
    )
ECHONET_ROOT = os.path.dirname(candidates[0])
VIDEOS_DIR = os.path.join(ECHONET_ROOT, "Videos")
if not os.path.isdir(VIDEOS_DIR):
    # some mirrors nest an extra folder level
    nested = glob.glob(os.path.join(ECHONET_ROOT, "**", "Videos"), recursive=True)
    VIDEOS_DIR = nested[0] if nested else VIDEOS_DIR
print("EchoNet root:", ECHONET_ROOT)
print("Videos dir:", VIDEOS_DIR)

## Official split

`FileList.csv` ships its own `Split` column (`TRAIN`/`VAL`/`TEST`) assigned by
the dataset's authors — use it as-is, the same discipline `docs/12` applies to
PTB-XL's folds and BraTS's official split.

In [ ]:
# Cell 4 — load metadata and the official split.
file_list = pd.read_csv(os.path.join(ECHONET_ROOT, "FileList.csv"))
print(file_list[["FileName", "EF", "Split"]].head())
print(file_list.Split.value_counts())

train_df = file_list[file_list.Split == "TRAIN"].reset_index(drop=True)
val_df = file_list[file_list.Split == "VAL"].reset_index(drop=True)
test_df = file_list[file_list.Split == "TEST"].reset_index(drop=True)
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")

## Dataset: sample frames per video

We sample `NUM_FRAMES` evenly-spaced frames per echo clip (covering at least
one full cardiac cycle in most EchoNet-Dynamic videos), resize to 112x112, and
normalize. This is the standard "decimate + frame-sample" compromise for video
regression on limited compute — full-framerate 3D convolution over the whole
clip is the more expensive alternative `docs/12` implicitly steers away from
for a single-GPU budget (same "patch/embedding over full-volume" principle
used for BraTS and CT-FM).

In [ ]:
# Cell 5 — Dataset class: reads an .avi, samples frames, normalizes.
NUM_FRAMES = 16
FRAME_SIZE = 112


def read_sampled_frames(video_path: str, num_frames: int, frame_size: int) -> np.ndarray:
    capture = cv2.VideoCapture(video_path)
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, max(total_frames - 1, 0), num_frames).astype(int)

    frames = []
    for target_index in frame_indices:
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(target_index))
        success, frame = capture.read()
        if not success:
            frame = np.zeros((frame_size, frame_size, 3), dtype=np.uint8)
        else:
            frame = cv2.resize(frame, (frame_size, frame_size))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    capture.release()

    clip = np.stack(frames).astype("float32") / 255.0  # (T, H, W, C)
    clip = (clip - 0.5) / 0.5
    return clip.transpose(3, 0, 1, 2)  # -> (C, T, H, W)


class EchoNetDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, videos_dir: str):
        self.dataframe = dataframe
        self.videos_dir = videos_dir

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, index: int):
        row = self.dataframe.iloc[index]
        filename = row.FileName if row.FileName.endswith(".avi") else f"{row.FileName}.avi"
        video_path = os.path.join(self.videos_dir, filename)
        clip = read_sampled_frames(video_path, NUM_FRAMES, FRAME_SIZE)
        ejection_fraction = np.float32(row.EF) / 100.0  # normalize EF% to [0, 1] for stable regression
        return torch.from_numpy(clip), torch.tensor(ejection_fraction)


train_ds = EchoNetDataset(train_df, VIDEOS_DIR)
val_ds = EchoNetDataset(val_df, VIDEOS_DIR)
test_ds = EchoNetDataset(test_df, VIDEOS_DIR)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

## Model: per-frame CNN encoder + temporal pooling

A frozen-ish ImageNet-pretrained ResNet18 (fine-tuned end-to-end here, since
echo frames look very different from natural images and full fine-tuning
converges better for this modality shift) encodes each sampled frame
independently; a small temporal attention-pool aggregates across frames before
the regression head — the same attention-pooling idea as the pathology MIL
notebook, applied along time instead of across patches.

In [ ]:
# Cell 6 — model definition.
class EchoEFRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        backbone.fc = nn.Identity()
        self.frame_encoder = backbone
        self.frame_feature_dim = 512

        self.temporal_attention = nn.Sequential(
            nn.Linear(self.frame_feature_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )
        self.regression_head = nn.Sequential(
            nn.Linear(self.frame_feature_dim, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
            nn.Sigmoid(),  # EF was normalized to [0, 1]
        )

    def forward(self, clip: torch.Tensor) -> torch.Tensor:
        # clip: (batch, C, T, H, W) -> per-frame encode -> (batch, T, feature_dim)
        batch_size, channels, num_frames, height, width = clip.shape
        frames = clip.permute(0, 2, 1, 3, 4).reshape(batch_size * num_frames, channels, height, width)
        frame_features = self.frame_encoder(frames).view(batch_size, num_frames, self.frame_feature_dim)

        attention_logits = self.temporal_attention(frame_features).squeeze(-1)  # (batch, T)
        attention_weights = torch.softmax(attention_logits, dim=1).unsqueeze(-1)  # (batch, T, 1)
        pooled_features = (attention_weights * frame_features).sum(dim=1)  # (batch, feature_dim)

        return self.regression_head(pooled_features).squeeze(-1)


model = EchoEFRegressor().to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

## Training loop

MAE loss directly on normalized EF (equivalent to MAE in EF-percentage terms
after multiplying back by 100) — matches `docs/12`'s "ejection-fraction
regression" task framing.

In [ ]:
# Cell 7 — train + validate.
criterion = nn.L1Loss()  # MAE
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)


@torch.no_grad()
def evaluate(loader: DataLoader) -> tuple[float, float]:
    model.eval()
    all_true, all_pred = [], []
    for clips, ef_true in loader:
        clips = clips.to(DEVICE)
        ef_pred = model(clips).cpu()
        all_true.append(ef_true * 100.0)
        all_pred.append(ef_pred * 100.0)
    y_true = torch.cat(all_true).numpy()
    y_pred = torch.cat(all_pred).numpy()
    return mean_absolute_error(y_true, y_pred), r2_score(y_true, y_pred)


NUM_EPOCHS = 10
best_val_mae = float("inf")
best_state_dict = None

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for clips, ef_true in train_loader:
        clips, ef_true = clips.to(DEVICE), ef_true.to(DEVICE)
        optimizer.zero_grad()
        ef_pred = model(clips)
        loss = criterion(ef_pred, ef_true)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * clips.size(0)

    train_loss = running_loss / len(train_ds)
    val_mae, val_r2 = evaluate(val_loader)
    print(f"epoch {epoch+1:02d}  train_mae_norm={train_loss:.4f}  val_EF_MAE={val_mae:.2f}  val_R2={val_r2:.4f}")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state_dict)
print("Best val EF MAE:", best_val_mae)

In [ ]:
# Cell 8 — held-out test evaluation.
test_mae, test_r2 = evaluate(test_loader)
print(f"Test EF MAE: {test_mae:.2f} percentage points")
print(f"Test R^2: {test_r2:.4f}")

## Save the checkpoint

In [ ]:
# Cell 9 — save checkpoint + metadata.
checkpoint_path = "/kaggle/working/echonet_ef_regressor.pt"
torch.save(model.state_dict(), checkpoint_path)

metadata = {
    "model": "EchoEFRegressor",
    "num_frames_sampled": NUM_FRAMES,
    "frame_size": FRAME_SIZE,
    "test_ef_mae": float(test_mae),
    "test_r2": float(test_r2),
    "trained_on": "EchoNet-Dynamic, official FileList.csv Split column",
}
with open("/kaggle/working/echonet_ef_regressor.json", "w") as handle:
    json.dump(metadata, handle, indent=2)

print("Saved:", checkpoint_path)
print(json.dumps(metadata, indent=2))

## Next steps (outside this notebook)

1. **Re-check Hugging Face for an Echo-Vision-FM-class checkpoint before
   committing to this training path** — per `docs/04`, this was unverified at
   planning time specifically because the space moves fast.
2. This notebook only does **EF regression**; `docs/12` also lists **wall-motion
   classification** as in-scope for this vertical — that's a separate
   multi-class head (or a separate model) over the same frame features, not
   built here.
3. Subgroup breakdown, calibration, and registry entry — same requirement as
   every other vertical (docs/06, docs/10).
4. If you used an unofficial Kaggle mirror for prototyping, **switch to the
   official Stanford EchoNet-Dynamic release** (with its data-use agreement)
   before anything beyond internal experimentation.